pydantic base model ()

class Prompter:

    def __init__(self, db_path: str, config:Dict [str]): (dbpath
        keep dbpath if end with .tmp else make dbpath with .tmp ending
        config = config
        create duckdb con
        make sure columns have right columns like status etc

    __exit__


    setup db
        check status column

    async send single prompt
        make sure status is not complete

    resume/start loop
        start at row with no status and gather them into transactions to do
        -- use ACID for sql transactions 
            Atomicity: A transaction is either fully completed or fully rolled back. If any part fails, the entire transaction fails.
            Consistency: A transaction moves the database from one valid state to another while following all rules and constraint.
            Isolation: Transactions run independently of each other, even when executed at the same time.
            Durability: Once a transaction is committed, its changes remain permanent, even after a system failure

    finalize
        change db file name from .duckdb.tmp -> .duckdb once all status's are complete
        


    
    

In [15]:
import asyncio
import duckdb
from pydantic import BaseModel, ValidationError, BeforeValidator, field_validator
from typing import Optional, List, Dict, Any
from pathlib import Path
from enum import StrEnum
import re

class Classification(StrEnum):
    BENIGN = 'Benign'
    LIKELY_BENIGN = 'Likely Benign'
    PATHOGENIC = 'Pathogenic'
    LIKELY_PATHOGENIC = 'Likely Pathogenic'
    VUS = 'VUS'

class Pathogenicity(BaseModel):
    classification: Classification

    @field_validator('classification', mode='before')
    @classmethod
    def parse(cls, value: str | Classification) -> str:
        if isinstance(value, Classification):
            return value

        text = str(value).lower().strip()

        mapping = {
            # '[ \-_]*' means zero or more spaces, dashes, or underscores
            r'variant[ \-_]*of[ \-_]*uncertain[ \-_]*significance': Classification.VUS,
            r'variant[ \-_]*of[ \-_]*unknown[ \-_]*significance': Classification.VUS,
            r'uncertain[ \-_]*significance': Classification.VUS,
            r'unknown[ \-_]*significance': Classification.VUS,
            
            r'likely[ \-_]*pathogenic': Classification.LIKELY_PATHOGENIC,
            r'likely[ \-_]*oncogenic': Classification.LIKELY_PATHOGENIC,
            
            r'likely[ \-_]*benign': Classification.LIKELY_BENIGN,
            
            # No regex needed for single words
            r'pathogenic': Classification.PATHOGENIC,
            r'oncogenic': Classification.PATHOGENIC,
            r'benign': Classification.BENIGN,
            r'vus': Classification.VUS,
        }

        for pattern, _classification in mapping.items():
            if re.search(rf'\b{pattern}\b', text):
                
                return _classification

        raise ValueError("No valid classification found.")
    
class LLMResponse(BaseModel):
    classification: Pathogenicity #= Field(...)

class Prompter:
    """
    This is a reST style.
    
    :param param1: this is a first param
    :param param2: this is a second param
    :returns: this is a description of what is returned
    :raises keyError: raises an exception
    """


    def __init__(self, db_path: str, config: Path):
        """
        This is a reST style.
        
        :param param1: this is a first param
        :param param2: this is a second param
        :returns: this is a description of what is returned
        :raises keyError: raises an exception
        """
        self.db_path = db_path if db_path.endswith(".tmp") else f"{db_path}.tmp"
        self.config = config
        self.conn = duckdb.connect(self.db_path)
        self._setup_db()

    def _setup_db(self):
        """
        This is a reST style.
        
        :param param1: this is a first param
        :param param2: this is a second param
        :returns: this is a description of what is returned
        :raises keyError: raises an exception
        """
        # SQL logic to ensure 'status' (pending/completed/failed) columns exist
        pass

    async def send_single_prompt(self, row_id: str, prompt: str) -> Optional[LLMResponse]:
        """
        This is a reST style.
        
        :param param1: this is a first param
        :param param2: this is a second param
        :returns: this is a description of what is returned
        :raises keyError: raises an exception
        """
        try:
            # Async HTTP request logic here
            # response = await client.post(...)
            
            # Runtime Validation
            # return LLMResponse.model_validate(response.json())
            pass
        except Exception as e:
            # Log error and return None so status can be marked 'failed'
            return None

    async def resume_loop(self):
        """
        This is a reST style.
        
        :param param1: this is a first param
        :param param2: this is a second param
        :returns: this is a description of what is returned
        :raises keyError: raises an exception
        """
        # 1. Query: SELECT * FROM data WHERE status != 'completed'
        # 2. Gather tasks for send_single_prompt
        # 3. Use Transactional Commits (BEGIN/COMMIT) for batch writes
        pass

    def finalize(self):
        """
        This is a reST style.
        
        :param param1: this is a first param
        :param param2: this is a second param
        :returns: this is a description of what is returned
        :raises keyError: raises an exception
        """
        pass

In [17]:
# --- Test Cases ---

# Case 1: The "No Spaces" / CamelCase LLM Hallucination
print(Pathogenicity(classification="LikelyPathogenic"))
# Output: classification=<Classification.LIKELY_PATHOGENIC: 'likelypathogenic'>

# Case 2: The "Forgot Spaces" inside a Paragraph
print(Pathogenicity(classification="Review suggests this is LikelyBenign based on frequency."))
# Output: classification=<Classification.LIKELY_BENIGN: 'likelybenign'>

# Case 3: Standard Spacing
print(Pathogenicity(classification="Likely Pathogenic"))
# Output: classification=<Classification.LIKELY_PATHOGENIC: 'likelypathogenic'>

# Case 4: Long Squashed Phrase
print(Pathogenicity(classification="variantofuncertainsignificance"))
# Output: classification=<Classification.VUS: 'vus'>

print(Pathogenicity(classification="Variant of uncertain significance"))

print(Pathogenicity(classification="Classification: 'Pathogenic'"))

classification=<Classification.LIKELY_PATHOGENIC: 'Likely Pathogenic'>
classification=<Classification.LIKELY_BENIGN: 'Likely Benign'>
classification=<Classification.LIKELY_PATHOGENIC: 'Likely Pathogenic'>
classification=<Classification.VUS: 'VUS'>
classification=<Classification.VUS: 'VUS'>
classification=<Classification.PATHOGENIC: 'Pathogenic'>
